# Visualizações da silver (AV1)

Este notebook complementa a demonstração técnica. Os gráficos aqui respondem, um a um, às sub-perguntas da pesquisa, sem mostrar nada que não sirva para contar a história.

**Pergunta central:**
> Como as ondas do COVID-19 impactaram o volume de internações e a mortalidade hospitalar no SUS-SP entre 2020 e 2023?

**Quatro blocos, um por sub-pergunta:**

1. **Volume mensal de internações.** Compara COVID com o restante. Mostra as ondas.
2. **Taxa de óbito hospitalar mensal.** Mostra se morreu mais gente nos picos.
3. **Perfil da internação.** Tempo internado e uso de UTI, COVID frente ao restante.
4. **Quem foi mais afetado.** Faixa etária e sexo dentro das internações COVID.

> Tudo é calculado em memória sobre a silver granular, em que cada linha segue sendo uma AIH. Nenhuma tabela agregada é salva em disco. A geração da camada gold fica para a AV2.

## 0. Configuração do ambiente

In [ ]:
import os
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

ROOT = Path(os.getcwd())
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SILVER_FILE = ROOT / 'data' / 'silver' / 'sihsus_sp.parquet'
print(f'Silver: {SILVER_FILE} | existe: {SILVER_FILE.exists()}')

In [ ]:
df = pd.read_parquet(SILVER_FILE)
print(f'Registros:                    {len(df):,}')
print(f'Colunas:                      {len(df.columns)}')
print(f'Período:                      {df["DT_INTER"].min().date()} a {df["DT_INTER"].max().date()}')
print(f'Internações COVID (CID B342): {df["is_covid"].sum():,} ({100*df["is_covid"].mean():.1f}% do total)')
df.head(3)

## 1. Volume mensal de internações: COVID frente ao restante

Este é o gráfico central da apresentação. Ele mostra, mês a mês, quantas internações foram por COVID (CID B342) e quantas foram por qualquer outro motivo. Os picos da série vermelha identificam as ondas: a primeira em 2020, a segunda no começo de 2021 (variante Gama) e a terceira no fim de 2021 e início de 2022 (variante Ômicron).

In [ ]:
por_mes = df.groupby('ano_mes').agg(
    total=('is_covid', 'size'),
    covid=('is_covid', 'sum'),
).reset_index()
por_mes['nao_covid'] = por_mes['total'] - por_mes['covid']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(por_mes['ano_mes'], por_mes['nao_covid'],
        label='Internações por outros motivos', color='steelblue', linewidth=2)
ax.plot(por_mes['ano_mes'], por_mes['covid'],
        label='Internações por COVID (CID B342)', color='crimson', linewidth=2)
ax.set_title('Internações mensais no SUS-SP, 2020 a 2023')
ax.set_xlabel('Mês')
ax.set_ylabel('Número de internações')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 2. Taxa de óbito hospitalar mensal: COVID frente ao restante

Mesma ideia do gráfico anterior, mas em vez de contagem mostramos a taxa de óbito. Cada linha representa a proporção de internações que terminaram em óbito, calculada mês a mês. A linha vermelha (COVID) tende a ficar sempre acima da azul (outros motivos), e a distância cresce nos picos da pandemia.

In [ ]:
tx_mes = (df.groupby(['ano_mes', 'is_covid'])['MORTE']
            .mean()
            .unstack(fill_value=0) * 100)
tx_mes = tx_mes.rename(columns={False: 'Outros motivos', True: 'COVID (B342)'})

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tx_mes.index, tx_mes['Outros motivos'],
        label='Outros motivos', color='steelblue', linewidth=2)
ax.plot(tx_mes.index, tx_mes['COVID (B342)'],
        label='COVID (B342)', color='crimson', linewidth=2)
ax.set_title('Taxa de óbito hospitalar por mês, SUS-SP (2020 a 2023)')
ax.set_xlabel('Mês')
ax.set_ylabel('Óbitos por 100 internações (%)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 3. Perfil da internação: tempo internado e uso de UTI

Este bloco compara duas coisas entre pacientes COVID e não-COVID:

- **Dias médios de internação:** em média, quanto tempo a pessoa ficou no hospital.
- **Proporção de internações que precisaram de UTI:** de cada 100 internações, quantas tiveram pelo menos um dia em UTI no mês.

Se o COVID tem um perfil mais grave, as barras vermelhas ficam maiores que as azuis nos dois gráficos.

In [ ]:
perfil = df.groupby('is_covid').agg(
    dias_medios=('DIAS_PERM', 'mean'),
    pct_uti=('UTI_MES_TO', lambda x: (x > 0).mean() * 100),
)
perfil.index = ['Outros motivos', 'COVID (B342)']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cores = ['steelblue', 'crimson']

b1 = axes[0].bar(perfil.index, perfil['dias_medios'], color=cores)
axes[0].set_title('Dias médios de internação')
axes[0].set_ylabel('Dias')
for bar, v in zip(b1, perfil['dias_medios']):
    axes[0].text(bar.get_x() + bar.get_width()/2, v,
                 f'{v:.1f}', ha='center', va='bottom', fontsize=10)

b2 = axes[1].bar(perfil.index, perfil['pct_uti'], color=cores)
axes[1].set_title('Internações que precisaram de UTI')
axes[1].set_ylabel('Porcentagem das internações (%)')
for bar, v in zip(b2, perfil['pct_uti']):
    axes[1].text(bar.get_x() + bar.get_width()/2, v,
                 f'{v:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Quem foi mais afetado: faixa etária e sexo

Os dois gráficos abaixo olham apenas para as internações COVID (CID B342) e mostram como elas se distribuíram pelos grupos demográficos. Eles respondem à sub-pergunta sobre quais grupos concentraram mais casos dentro do recorte de São Paulo.

In [ ]:
covid_df = df[df['is_covid']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

por_idade = covid_df['faixa_etaria'].value_counts().sort_index()
axes[0].bar(por_idade.index, por_idade.values, color='crimson')
axes[0].set_title('Internações COVID por faixa etária')
axes[0].set_xlabel('Faixa etária (anos)')
axes[0].set_ylabel('Número de internações')
for i, v in enumerate(por_idade.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

mapa_sexo = {'1': 'Masculino', '3': 'Feminino'}
por_sexo = (covid_df['SEXO'].map(mapa_sexo)
                            .fillna('Não informado')
                            .value_counts())
axes[1].bar(por_sexo.index, por_sexo.values, color='crimson')
axes[1].set_title('Internações COVID por sexo')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Número de internações')
for i, v in enumerate(por_sexo.values):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Observação final

Os quatro blocos acima cobrem as sub-perguntas da pesquisa da AV1. Nenhuma agregação é salva em disco, tudo é calculado em memória sobre a silver granular. As análises consolidadas (mortalidade por onda, uso de UTI por onda, impacto regional, modelagem) ficam para a AV2, junto com a camada gold.